In [1]:
def pagerank(G, alpha=0.85, personalization=None, max_iter=100, tol=1.0e-6, nstart=None, weight='weight', dangling=None):
    if len(G) == 0: 
        return {} 
  
    if not G.is_directed(): 
        D = G.to_directed() 
    else: 
        D = G
        
    # Create a copy in (right) stochastic form 
    W = nx.stochastic_graph(D, weight=weight) 
    N = W.number_of_nodes()
    
    # Choose fixed starting vector if not given 
    if nstart is None: 
        x = dict.fromkeys(W, 1.0 / N) 
    else: 
        # Normalized nstart vector 
        s = float(sum(nstart.values())) 
        x = dict((k, v / s) for k, v in nstart.items()) 
        
    if personalization is None:
        # Assign uniform personalization vector if not given 
        p = dict.fromkeys(W, 1.0 / N)
    else: 
        missing = set(G) - set(personalization) 
        if missing: 
            raise NetworkXError('Personalization dictionary must have a value for every node. Missing nodes %s' % missing) 
        s = float(sum(personalization.values())) 
        p = dict((k, v / s) for k, v in personalization.items())
        
    if dangling is None:
        # Use personalization vector if dangling vector not specified 
        dangling_weights = p 
    else: 
        missing = set(G) - set(dangling) 
        if missing: 
            raise NetworkXError('Dangling node dictionary must have a value for every node. Missing nodes %s' % missing) 
        s = float(sum(dangling.values())) 
        dangling_weights = dict((k, v/s) for k, v in dangling.items())
        
    dangling_nodes = [n for n in W if W.out_degree(n, weight=weight) == 0.0]
    
     # power iteration: make up to max_iter iterations 
    for _ in range(max_iter): 
        xlast = x 
        x = dict.fromkeys(xlast.keys(), 0) 
        danglesum = alpha * sum(xlast[n] for n in dangling_nodes) 
        for n in x:
            # this matrix multiply looks odd because it is 
            # doing a left multiply x^T=xlast^T*W 
            for nbr in W[n]: 
                x[nbr] += alpha * xlast[n] * W[n][nbr][weight] 
            x[n] += danglesum * dangling_weights[n] + (1.0 - alpha) * p[n] 
  
        # check convergence, l1 norm 
        err = sum([abs(x[n] - xlast[n]) for n in x]) 
        if err < N*tol: 
            return x 
    raise NetworkXError('Pagerank: power iteration failed to converge in %d iterations.' % max_iter)

In [3]:
import networkx as nx

In [5]:
G = nx.barabasi_albert_graph(60, 41) 
pr = nx.pagerank(G, 0.4)

In [7]:
print(pr)

{0: 0.028095117224661056, 1: 0.013576414386402677, 2: 0.013378539895842962, 3: 0.012570045358309027, 4: 0.0127597524073768, 5: 0.01298789213997, 6: 0.013164812953466446, 7: 0.012969474307807044, 8: 0.013175048517700952, 9: 0.013169556284592323, 10: 0.012766414087056964, 11: 0.012558119827373072, 12: 0.013167882876591836, 13: 0.01336054892944602, 14: 0.012768556264449706, 15: 0.012956663277157096, 16: 0.01275745201309511, 17: 0.013166256904496847, 18: 0.012776837020780281, 19: 0.01215236965418224, 20: 0.012967098973762739, 21: 0.012755370114743935, 22: 0.01336929194938491, 23: 0.012960855852917893, 24: 0.013565281606302696, 25: 0.013364737259372171, 26: 0.012169194078296402, 27: 0.01235180249547408, 28: 0.01316784443347516, 29: 0.01336054892944602, 30: 0.012754406013697666, 31: 0.01277642158066579, 32: 0.013567482725350648, 33: 0.013369286627791336, 34: 0.012962958870810084, 35: 0.012574039763768823, 36: 0.012948404236661992, 37: 0.013575195412284278, 38: 0.01217491374619492, 39: 0.0123